# Stochastic dynamical systems: examples, validation and ablations

Affine saddle, gLV heteroclinic channel and two-saddle sphere (Section 3, Fig. 2), CCA benchmark (Appendix F, Fig. 4), model selection (Appendix J, Fig. 6) and ablations (Appendix L, Fig. 9).

In [ ]:
import itertools
import os
import re

import matplotlib
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from matplotlib.collections import LineCollection
from mpl_toolkits.mplot3d.art3d import Line3DCollection
from scipy.optimize import linear_sum_assignment
from sklearn.cross_decomposition import CCA
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
from sklearn.preprocessing import StandardScaler
from sklearn.utils.extmath import randomized_svd

RESULTS_DIR = "results"  # output folder for figures
DATA_DIR = "data"  # output folder for intermediate results
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)

In [ ]:
plt.style.use("tableau-colorblind10")
plt.rcParams.update(
    {
        "figure.autolayout": True,
        "font.size": 10,
        "axes.titlesize": 10,
        "axes.labelsize": 10,
        "xtick.labelsize": 10,
        "ytick.labelsize": 10,
        "font.family": "serif",
        "font.serif": "cmr10",
        "mathtext.fontset": "cm",
        "axes.formatter.use_mathtext": True,
    }
)

In [ ]:
def finite_difference_vectors(x):
    finite_diff = np.diff(x.T, axis=0)

    X = finite_diff[:-1, :].T
    Y = finite_diff[1:, :].T

    return X, Y


def normalize_Z(Z, axis=1, eps=1e-10):
    norm = np.linalg.norm(Z, axis=axis, keepdims=True)
    norm = np.maximum(norm, eps)
    return Z / norm


def nnsvd(X_init, k, eps: float = 1e-6, random_state=None):
    m, n = X_init.shape
    r = min(k, m, n)

    U, S, V = randomized_svd(X_init, n_components=r, random_state=random_state)

    W = np.zeros((m, k))
    H = np.zeros((k, n))

    W[:, 0] = np.sqrt(S[0]) * np.abs(U[:, 0])
    H[0, :] = np.sqrt(S[0]) * np.abs(V[0, :])

    for j in range(1, r):
        x, y = U[:, j], V[j, :]

        x_p, y_p = np.maximum(x, 0), np.maximum(y, 0)
        x_n, y_n = np.abs(np.minimum(x, 0)), np.abs(np.minimum(y, 0))

        x_p_nrm, y_p_nrm = np.linalg.norm(x_p), np.linalg.norm(y_p)
        x_n_nrm, y_n_nrm = np.linalg.norm(x_n), np.linalg.norm(y_n)

        m_p = x_p_nrm * y_p_nrm
        m_n = x_n_nrm * y_n_nrm

        if m_p > m_n:
            u = x_p / (x_p_nrm + eps)
            v = y_p / (y_p_nrm + eps)
            sigma = m_p
        else:
            u = x_n / (x_n_nrm + eps)
            v = y_n / (y_n_nrm + eps)
            sigma = m_n

        lbd = np.sqrt(S[j] * sigma)
        W[:, j] = lbd * u
        H[j, :] = lbd * v

    if k > r:
        rng = (
            np.random.default_rng(random_state)
            if not isinstance(random_state, np.random.Generator)
            else random_state
        )

        scale = np.mean(W[:, :r]) if r > 0 else 1.0
        if scale == 0:
            scale = 1.0

        W[:, r:] = rng.random((m, k - r)) * scale * 0.1
        H[r:, :] = rng.random((k - r, n)) * scale * 0.1

    W[W < eps] = 0
    H[H < eps] = 0

    return W, H

## LPD algorithms

In [ ]:
# Offline LPD (Algorithm 2)
def lpd_offline(
    X,
    Y,
    k=1,
    W_init=None,
    M_init=None,
    relu=False,
    eps=1e-5,
    Nmax=2_000,
    Nz_max=100,
    toll=1e-10,
    eta_z=0.4,
    eta_W=0.1,
    eta_M=0.2,
    w=1,
):
    T = X.shape[1]
    Nx, Ny = X.shape[0], Y.shape[0]

    XG = (
        X.T
        @ np.linalg.inv(X @ X.T + eps * np.eye(Nx))
        @ X
        @ Y.T
        @ np.linalg.inv(Y @ Y.T + eps * np.eye(Ny))
        @ Y
        @ X.T
        @ np.linalg.inv(X @ X.T + eps * np.eye(Nx))
    )

    if M_init is None:
        M_init = np.eye(k)
        M = M_init
    else:
        M = M_init

    if W_init is None:
        W_init, _ = nnsvd(np.tile(X, (1, k)), k, eps=eps)
        W_init = W_init.T
        W = W_init
    else:
        W = W_init

    Z = np.random.rand(k, T)
    if relu:
        Z = np.maximum(Z, 0)
    Z = normalize_Z(Z, axis=1)

    err_list = []
    obs_list = []

    for n in range(Nmax):

        if relu:
            for _ in range(Nz_max):
                Z_old = Z.copy()
                Z = np.maximum(Z + eta_z * (W @ X - M @ Z), 0)

                if (
                    np.linalg.norm(Z - Z_old, ord="fro")
                    / (np.linalg.norm(Z_old, ord="fro") + 1e-12)
                    < toll
                ):
                    break
        else:
            Z = np.linalg.solve(M + eps * np.eye(k), W @ X)

        Z = normalize_Z(Z, axis=1)

        M_new = M + eta_M * (1 / T * (Z @ Z.T) - M)
        W_new = W + eta_W * (1 / T * Z @ XG - W)

        err_M = np.linalg.norm(M_new - M, ord="fro") / (
            np.linalg.norm(M, ord="fro") + 1e-12
        )
        err_W = np.linalg.norm(W_new - W, ord="fro") / (
            np.linalg.norm(W, ord="fro") + 1e-12
        )

        M = w * M_new + (1 - w) * M_init
        W = w * W_new + (1 - w) * W_init

        err_list.append((err_M, err_W))
        obs_list.append((np.linalg.norm(M, ord="fro"), np.linalg.norm(W, ord="fro")))

        if err_M < toll and err_W < toll:
            break

        if n == Nmax - 1:
            print("Warning: maximum number of iterations reached.")

    return Z, (W, M), (err_list, obs_list)


# Online LPD (Algorithm 1): running covariances, rectified inference
def lpd_online(
    X,
    Y,
    k=1,
    W_init=None,
    M_init=None,
    relu=False,
    eps=1e-5,
    Nz_max=100,
    toll=1e-10,
    eta_z=0.4,
    w=1,
):
    T = X.shape[1]
    Nx, Ny = X.shape[0], Y.shape[0]

    if M_init is None:
        M_init = np.eye(k)
        M = M_init
    else:
        M = M_init

    if W_init is None:
        W_init, _ = nnsvd(np.tile(X, (1, k)), k, eps=eps)
        W_init = W_init.T
        W = W_init
    else:
        W = W_init

    Cxx = np.zeros((Nx, Nx))
    Cyy = np.zeros((Ny, Ny))
    Cxy = np.zeros((Nx, Ny))

    Czx = np.zeros((k, Nx))
    Czz = np.zeros((k, k))

    Z = np.zeros((k, T))

    err_list = []
    obs_list = []

    for t in range(T):

        x_t = X[:, t : t + 1]
        y_t = Y[:, t : t + 1]

        Cxx = Cxx * t / (t + 1) + (x_t @ x_t.T) / (t + 1)
        Cyy = Cyy * t / (t + 1) + (y_t @ y_t.T) / (t + 1)
        Cxy = Cxy * t / (t + 1) + (x_t @ y_t.T) / (t + 1)

        Cxx_inv = np.linalg.inv(Cxx + eps * np.eye(Nx))
        Cyy_inv = np.linalg.inv(Cyy + eps * np.eye(Ny))

        G = Cxx_inv @ Cxy @ Cyy_inv @ Cxy.T @ Cxx_inv

        z_t = np.ones((k, 1)) + np.random.randn(k, 1) * 0.1

        if t == 0:
            Czx = z_t @ x_t.T
            Czz = z_t @ z_t.T

        if relu:
            for _ in range(Nz_max):
                z_old = z_t.copy()
                z_t = np.maximum(z_t + eta_z * (W @ x_t - M @ z_t), 0)

                if (
                    np.linalg.norm(z_old - z_t, ord="fro")
                    / (np.linalg.norm(z_t, ord="fro") + 1e-12)
                    < toll
                ):
                    break
        else:
            z_t = np.linalg.solve(M + eps * np.eye(k), W @ x_t)

        Z[:, t] = z_t.ravel()

        Czx = Czx * t / (t + 1) + (z_t @ x_t.T) / (t + 1)
        Czz = Czz * t / (t + 1) + (z_t @ z_t.T) / (t + 1)

        M_old = M.copy()
        W_old = W.copy()

        M = w * Czz + (1 - w) * M_init
        W = w * Czx @ G + (1 - w) * W_init

        err_M = np.linalg.norm(M_old - M, ord="fro")
        err_W = np.linalg.norm(W_old - W, ord="fro") / (
            np.linalg.norm(W, ord="fro") + 1e-8
        )

        err_list.append((err_M, err_W))
        obs_list.append((np.linalg.norm(M, ord="fro"), np.linalg.norm(W, ord="fro")))

        if err_M < toll and err_W < toll:
            break

    return Z, (W, M), (err_list, obs_list)

## Affine saddle (Fig. 2a)

In [ ]:
# Affine saddle dynamics (Appendix I.1)
def simulate_2d_non_normal_saddle(
    T=100,
    lambda_u=0.5,
    lambda_s=0.5,
    dt=0.1,
    kappa=0,
    sigma=0.05,
    x_star=np.array([0.0, 1.0]),
    x0=np.array([0.0, 4.0]),
    seed=0,
):
    rng = np.random.default_rng(seed)

    A = np.array(
        [
            [np.exp(lambda_u * dt), kappa],
            [0.0, np.exp(-lambda_s * dt)],
        ]
    )

    X = np.zeros((T, 2))
    X[0] = x0

    for t in range(T - 1):
        noise = sigma * rng.normal(size=2)
        X[t + 1] = x_star + A @ (X[t] - x_star) + noise

    return X, A

x_star = np.array([-1.0, 1.0])
lambda_u = 0.5
lambda_s = 0.5
kappa = 0
sigma = 0.05
T = 200
N = 48

X0_list = [[-1.1, 4], [-1.1, -4], [-0.9, 4], [-0.9, -4]]

A = np.array(
    [
        [np.exp(lambda_u * 0.1), kappa],
        [0.0, np.exp(-lambda_s * 0.1)],
    ]
)

eigvals, eigvecs = np.linalg.eig(A)
unstable_idx = np.argmax(np.abs(eigvals))
stable_idx = 1 - unstable_idx

u_unstable = eigvecs[:, unstable_idx].real
u_stable = eigvecs[:, stable_idx].real

u_unstable /= np.linalg.norm(u_unstable)
u_stable /= np.linalg.norm(u_stable)

X_fd_list = []
Y_fd_list = []
X_list = []

for n in range(N):
    X, _ = simulate_2d_non_normal_saddle(
        T=T,
        lambda_u=lambda_u,
        lambda_s=lambda_s,
        kappa=kappa,
        sigma=sigma,
        x_star=x_star,
        x0=[-1, np.random.uniform(-4, 4)],
        seed=18212 + n,
    )
    X_list.append(X)

    X_fd, Y_fd = finite_difference_vectors(X.T)
    X_fd_list.append(X_fd)
    Y_fd_list.append(Y_fd)

X0_list = [[-1.1, 4], [-1.1, -4], [-0.9, 4], [-0.9, -4]]
X_list_plot = []
for n in range(4):
    X, _ = simulate_2d_non_normal_saddle(
        T=T,
        lambda_u=lambda_u,
        lambda_s=lambda_s,
        kappa=kappa,
        sigma=0.0001,
        x_star=x_star,
        x0=X0_list[n],
        seed=18212 + n,
    )
    X_list_plot.append(X)


def vector_field(Xg, Yg, A, x_star):
    dx = Xg - x_star[0]
    dy = Yg - x_star[1]

    U = (A[0, 0] - 1.0) * dx + A[0, 1] * dy
    V = A[1, 0] * dx + (A[1, 1] - 1.0) * dy

    return U, V

plt.figure(figsize=(12 / 2.54, 12 / 2.54))

xv = np.linspace(-4, 4, 10)
yv = np.linspace(-4, 4, 10)
Xg, Yg = np.meshgrid(xv, yv)

U, V = vector_field(Xg, Yg, A, x_star)

plt.quiver(
    Xg,
    Yg,
    U,
    V,
    color="0.75",
    angles="xy",
    scale_units="xy",
    scale=0.3,
    width=0.005,
    alpha=0.8,
    zorder=1,
)

for X in X_list:
    plt.plot(X[:, 0], X[:, 1], color="black", alpha=0.35, lw=1.2, zorder=3)

plt.scatter(*x_star, color="black", s=60, zorder=5)

L = 2.2

plt.arrow(
    *x_star,
    *(L * u_unstable),
    color="crimson",
    width=0.01,
    head_width=0.12,
    length_includes_head=True,
    zorder=6,
)
plt.arrow(
    *x_star,
    *(-L * u_unstable),
    color="crimson",
    width=0.01,
    head_width=0.12,
    length_includes_head=True,
    zorder=6,
)

plt.arrow(
    *x_star + (0, L),
    *(-L * u_stable),
    color="royalblue",
    width=0.01,
    head_width=0.12,
    length_includes_head=True,
    zorder=6,
)
plt.arrow(
    *x_star - (0, L),
    *(L * u_stable),
    color="royalblue",
    width=0.01,
    head_width=0.12,
    length_includes_head=True,
    zorder=6,
)

plt.xlabel(r"$x_1$", fontsize=12)
plt.ylabel(r"$x_2$", fontsize=12)

plt.xlim(-4, 4)
plt.ylim(-4, 4)
plt.xticks([-4, 0, 4], fontsize=12)
plt.yticks([-4, 0, 4], fontsize=12)

plt.gca().set_aspect("equal", adjustable="box")
plt.gca().spines["top"].set_visible(False)
plt.gca().spines["right"].set_visible(False)
plt.tick_params(direction="out")

plt.grid(False)
plt.tight_layout()
plt.show()

In [ ]:
ONLINE = True
OFFLINE = not ONLINE
eps = 1e-5
k = 2

if OFFLINE:
    print("OFFLINE LPD")

    for i, (X_fd, Y_fd) in enumerate(zip(X_fd_list, Y_fd_list)):
        if i % 10 == 0:
            print("Sample   {}:".format(i), "out of:", len(X_fd_list))
        if i == 0:
            W_init, M_init = None, None
        else:
            W_init, M_init = W, M

        Z, (W, M), (err, obs) = lpd_offline(
            X_fd,
            Y_fd,
            k=k,
            relu=True,
            eps=eps,
            Nmax=1000,
            Nz_max=50,
            W_init=W_init,
            M_init=M_init,
            w=1 / (i + 1),
            toll=1e-10,
            eta_z=0.4,
            eta_W=0.4,
            eta_M=0.2,
        )

if ONLINE:
    print("ONLINE LPD")

    for i, (X_fd, Y_fd) in enumerate(zip(X_fd_list[:48], Y_fd_list[:48])):
        if i % 10 == 0:
            print("Sample   {}:".format(i), "out of:", len(X_fd_list))
        if i == 0:
            W_init, M_init = None, None
        else:
            W_init, M_init = W, M

        Z, (W, M), (err, obs) = lpd_online(
            X_fd,
            Y_fd,
            k=k,
            relu=True,
            eps=eps,
            Nz_max=200,
            W_init=W_init,
            M_init=M_init,
            w=1 / (i + 1),
            toll=1e-10,
            eta_z=0.2,
        )

In [ ]:
MW = np.linalg.inv(M + eps * np.eye(k)) @ W
MW = MW / np.linalg.norm(MW, axis=1, keepdims=True)

plt.figure(figsize=(12 / 2.54, 10 / 2.54))

plt.imshow(MW, cmap="bwr", aspect="auto")
plt.colorbar(label="Weight Value")
plt.xticks(
    ticks=np.arange(k), labels=[r"$x_{}$".format(i + 1) for i in range(k)], fontsize=12
)
plt.yticks(
    ticks=np.arange(MW.shape[0]),
    labels=[f"N{i+1}" for i in range(MW.shape[0])],
    fontsize=12,
)

plt.xlabel("Input Dimension", fontsize=12)
plt.ylabel("Output Dimension", fontsize=12)
plt.show()

plt.figure(figsize=(12 / 2.54, 10 / 2.54))
colors_quiver = ["royalblue", "firebrick"]
for i in range(k):
    plt.quiver(
        0,
        0,
        MW[i, 0],
        MW[i, 1],
        angles="xy",
        scale_units="xy",
        scale=1,
        label=f"N{i+1}",
        color=colors_quiver[i],
        width=0.01,
    )
plt.legend(fontsize=12)
plt.xlim(-1, 1)
plt.ylim(-1, 1)
plt.xlabel(r"$\Delta x_1$", fontsize=12)
plt.ylabel(r"$\Delta x_2$", fontsize=12)
plt.xticks([-1, 0, 1], fontsize=12)
plt.yticks([-1, 0, 1], fontsize=12)
plt.show()

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot([e[0] for e in err], label="err_M")
ax[0].plot([e[1] for e in err], label="err_W")
ax[0].legend()
ax[1].plot([o[0] for o in obs], label="||M||_F")
ax[1].plot([o[1] for o in obs], label="||W||_F")
ax[1].legend()
plt.show()

In [ ]:
theta = 0.3
beta = 10
alpha_min = 0.2
alpha_max = 1.0

traj_ids = [0, 1, 2, 3]
inset_ids = [1, 3]

MW = np.linalg.inv(M + eps * np.eye(k)) @ W
MW = MW / np.linalg.norm(MW, axis=1, keepdims=True)

v1 = MW[0, :]
v2 = MW[1, :]

colors = ["royalblue", "firebrick", "darkorange", "indigo", "goldenrod", "forestgreen"]

fig, ax0 = plt.subplots(figsize=(8 / 2.54, 8 / 2.54))

xv = np.linspace(-3.9, 4.1, 12)
yv = np.linspace(-4.1, 3.9, 12)
Xg, Yg = np.meshgrid(xv, yv)
U, V = vector_field(Xg, Yg, A, x_star)

ax0.quiver(
    Xg,
    Yg,
    U,
    V,
    color="0.75",
    angles="xy",
    scale_units="xy",
    scale=0.3,
    width=0.005,
    alpha=0.9,
    zorder=0,
)

trajectory_labels = ["(i)", "(ii)"]
label_offsets = [[-1.5, 3.5], [1, 3.5]]

for i in traj_ids:
    X = X_list_plot[i].T
    X = X[:, (X[0] < 4.1) & (X[1] < 4.1)]
    X = X[:, (X[0] > -4.5) & (X[1] > -4)]
    X = X.T

    dX = np.diff(X.T, axis=1)
    dX /= np.linalg.norm(dX, axis=0, keepdims=True)

    Z = np.maximum(MW @ dX, 0)
    Z /= np.max(np.abs(Z))

    label = np.argmax(Z, axis=0)
    magnitude = Z[label, np.arange(Z.shape[1])]

    m_act = 1 / (1 + np.exp(-beta * (np.abs(magnitude) - theta)))
    m_act = np.clip(m_act**0.5, alpha_min, alpha_max)

    points = X[:-1].reshape(-1, 1, 2)
    segments = np.concatenate([points[:-1], points[1:]], axis=1)

    colors_mapped = [
        matplotlib.colors.to_rgba(colors[l], alpha=a) for l, a in zip(label, m_act)
    ]

    lc = LineCollection(
        segments, colors=colors_mapped, linewidths=4.0, antialiased=False, zorder=5
    )
    ax0.add_collection(lc)

    if i in inset_ids:
        idx = inset_ids.index(i)
        dx, dy = label_offsets[idx]
        ax0.text(
            X[0, 0] + dx,
            X[0, 1] + dy,
            trajectory_labels[idx],
            fontsize=11,
            fontweight="bold",
            color="black",
        )

ax0.scatter(*x_star, marker="x", color="black", s=60, zorder=7)

inset_positions = [[0.05, 0.15, 0.28, 0.20], [0.65, 0.15, 0.28, 0.20]]

for idx, (pos, i) in enumerate(zip(inset_positions, inset_ids)):
    ax_in = ax0.inset_axes(pos)

    X = X_list_plot[i].T
    X = X[:, (X[0] < 4.1) & (X[1] < 4.1)]
    X = X[:, (X[0] > -4) & (X[1] > -4)]
    X = X.T

    dX = np.diff(X.T, axis=1)
    dX /= np.linalg.norm(dX, axis=0, keepdims=True)

    Z_latent = dX.T @ u_unstable
    Z_latent /= np.max(np.abs(Z_latent))

    Z_on_relu = np.maximum(MW @ dX, 0)
    Z_on_relu /= np.max(np.abs(Z_on_relu))

    ax_in.plot(Z_on_relu[0], lw=4, color=colors[0], alpha=0.9)
    ax_in.plot(Z_on_relu[1], lw=4, color=colors[1], alpha=0.9)
    ax_in.plot(-Z_latent if idx == 0 else Z_latent, color="black", lw=1, linestyle="--")

    ax_in.set_xticks([])
    ax_in.set_yticks([])
    ax_in.set_xlabel("time")

    ax_in.text(
        -0.18,
        0.5,
        r"$\mathbf{z}$",
        transform=ax_in.transAxes,
        fontsize=11,
        rotation=90,
        va="center",
    )

    ax_in.set_ylim(-0.05, 1.05)

    ax_in.spines["top"].set_visible(False)
    ax_in.spines["right"].set_visible(False)

    ax_in.text(
        0.02,
        0.96,
        trajectory_labels[idx],
        transform=ax_in.transAxes,
        fontsize=10,
        fontweight="bold",
        verticalalignment="top",
        zorder=10,
    )

ax_q = ax0.inset_axes([0.78, 0.78, 0.2, 0.2])
colors_quiver = ["royalblue", "firebrick"]

for i in range(k):
    ax_q.quiver(
        0,
        0,
        MW[i, 0],
        MW[i, 1],
        angles="xy",
        scale_units="xy",
        scale=1,
        color=colors_quiver[i],
        width=0.06,
    )

ax_q.set_xlim(-1, 1)
ax_q.set_ylim(-1, 1)
ax_q.set_aspect("equal")
ax_q.set_xticks([])
ax_q.set_yticks([])
ax_q.spines["top"].set_visible(False)
ax_q.spines["right"].set_visible(False)

ax_q.text(
    0.5,
    0.7,
    r"$V$",
    transform=ax_q.transAxes,
    fontsize=10,
    fontweight="bold",
    ha="center",
)
ax_q.text(
    0.35,
    -0.06,
    r"$\delta x_1$",
    transform=ax_q.transAxes,
    fontsize=11,
    fontweight="bold",
    va="top",
)
ax_q.text(
    -0.35,
    0.7,
    r"$\delta x_2$",
    transform=ax_q.transAxes,
    rotation=90,
    fontsize=11,
    fontweight="bold",
    va="top",
)

ax0.text(
    -0.15, 0.5, r"$x_2$", transform=ax0.transAxes, fontsize=12, rotation=90, va="center"
)
ax0.text(0.5, -0.15, r"$x_1$", transform=ax0.transAxes, fontsize=12, ha="center")

ax0.set_xlim(-4.5, 4)
ax0.set_xticks([-4, 0, 4])
ax0.set_ylim(-4, 4)
ax0.set_yticks([-4, 0, 4])
ax0.set_aspect("equal", adjustable="box")
ax0.spines["top"].set_visible(False)
ax0.spines["right"].set_visible(False)
ax0.tick_params(direction="out")
ax0.grid(False)

plt.savefig(
    os.path.join(RESULTS_DIR, "fig2a_saddle.png"),
    dpi=1000,
    bbox_inches="tight",
    pad_inches=0.0,
)
plt.show()

## gLV heteroclinic channel (Fig. 2b)

In [ ]:
T = 2000
dt = 0.1
sigma = 0.001

N_shc = 3
A_shc = np.ones((N_shc, N_shc)) * 1.5
np.fill_diagonal(A_shc, 1.0)

A_shc[0, 1] = 0.7
A_shc[1, 2] = 0.7
A_shc[2, 0] = 0.7

A = A_shc
N = N_shc

plt.figure(figsize=(12 / 2.54, 12 / 2.54))
im = plt.imshow(A, cmap="cividis", interpolation="nearest")

plt.xticks(range(N), [f"S{i+1}" for i in range(N)], fontsize=12)
plt.yticks(range(N), [f"S{j+1}" for j in range(N)], fontsize=12)

cbar = plt.colorbar(im)
cbar.set_label("Interaction Strength", fontsize=12)
cbar.set_ticks([0.7, 1, 1.5])
cbar.ax.tick_params(labelsize=12)

plt.tight_layout()
plt.show()

x0 = [0.5, 0.5, 0.5]

num_steps = int(T / dt)
X = np.zeros((N, num_steps))
X[:, 0] = x0

for k in range(1, num_steps):

    dx = X[:, k - 1] * (1 - A @ X[:, k - 1])

    dx += sigma * np.random.randn(N)

    X[:, k] = X[:, k - 1] + dx * dt

    X[:, k] = np.clip(X[:, k], 0, None)

trajs = X

t_eval = np.arange(0, T, dt)

plt.figure(figsize=(16 / 2.54, 5 / 2.54))
for i in range(N_shc):
    plt.plot(t_eval, X[i], label=f"S{i+1}")
for j in range(N_shc, N):
    plt.plot(t_eval, X[j], color="gray", alpha=0.3)
plt.xlabel("Time")
plt.xlim(800, 1000)
plt.ylabel("Population")
plt.xticks([])
plt.yticks([0, 1])
plt.legend()
plt.show()

plt.figure()
plt.plot(trajs.T)
plt.show()

N = 3

A = np.ones((N, N)) * 1.5
np.fill_diagonal(A, 1.0)
A[0, 1] = 0.7
A[1, 2] = 0.7
A[2, 0] = 0.7


def glv_velocity(x):
    return x * (1 - A @ x)

n_plane = np.array([1.0, 1.0, 1.0])
n_plane /= np.linalg.norm(n_plane)

s = np.linspace(0.0, 1, 10)
pts = []

for x in s:
    for y in s:
        z = 1.0 - x - y
        if z > 0:
            pts.append([x, y, z])

pts = np.array(pts)

V = np.array([glv_velocity(p) for p in pts])

V_plane = V - (V @ n_plane)[:, None] * n_plane

norms = np.linalg.norm(V_plane, axis=1, keepdims=True)
mask = norms.squeeze() > 1e-6

pts = pts[mask]
V_plane = V_plane[mask] / norms[mask]

fig = plt.figure(figsize=(8 / 2.54, 8 / 2.54))
ax = fig.add_subplot(111, projection="3d")

ax.quiver(
    pts[:, 0],
    pts[:, 1],
    pts[:, 2],
    V_plane[:, 0],
    V_plane[:, 1],
    V_plane[:, 2],
    length=0.15,
    normalize=True,
    linewidth=0.7,
    color="0.4",
    arrow_length_ratio=0.35,
)

stride = 10
ax.plot(
    trajs[0, ::stride],
    trajs[1, ::stride],
    trajs[2, ::stride],
    color="firebrick",
    lw=2.2,
    alpha=0.95,
)

saddles = np.eye(3)
ax.scatter(saddles[:, 0], saddles[:, 1], saddles[:, 2], color="black", s=40, zorder=5)

ax.set_xlim(0, 1.05)
ax.set_ylim(0, 1.05)
ax.set_zlim(0, 1.05)

ax.set_box_aspect([1, 1, 1])

ax.set_xticks([])
ax.set_yticks([])
ax.set_zticks([])

ax.set_xlabel(r"$x_1$")
ax.set_ylabel(r"$x_2$")
ax.set_zlabel(r"$x_3$")

ax.view_init(elev=25, azim=45)

ax.grid(False)

plt.tight_layout()

plt.show()

In [ ]:
N_stat = 10_000
X_fd, Y_fd = finite_difference_vectors(trajs[:, N_stat:])
N_data = 8_000
X = X_fd[:, :N_data]
Y = Y_fd[:, :N_data]

k = 3
ONLINE = False
OFFLINE = not ONLINE

if OFFLINE:
    print("Running offline LPD...")
    Z, (W_lpd, M_lpd), (err_list, obs_list) = lpd_offline(
        X,
        Y,
        k=k,
        relu=True,
        eps=1e-5,
        Nmax=3000,
        Nz_max=100,
        toll=1e-10,
        eta_z=0.4,
        eta_W=0.4,
        eta_M=0.2,
    )
else:
    print("Running online LPD...")
    Z, (W_lpd, M_lpd), (err_list, obs_list) = lpd_online(
        X,
        Y,
        k=k,
        relu=True,
        Nz_max=200,
        toll=1e-10,
        eps=1e-5,
        eta_z=0.2,
    )

plt.figure(figsize=(10, 5))
plt.plot(Z.T)
plt.xlabel("Time")
plt.ylabel("Latent Components")
plt.legend([f"Component {i+1}" for i in range(Z.shape[0])])
plt.show()

for i in range(Z.shape[0]):
    Z[i, :] = Z[i, :] / np.linalg.norm(Z[i, :])
G_exp = Z.T @ Z / Z.shape[1]
plt.figure(figsize=(6, 6))
plt.imshow(G_exp, cmap="viridis", interpolation="nearest")
plt.colorbar(label="G_exp Values")
plt.title("Expected Correlation Matrix G_exp from LPD")
plt.show()

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot([e[0] for e in err_list], label="err_M")
ax[0].plot([e[1] for e in err_list], label="err_W")
ax[0].legend()
ax[0].set_title("LPD optimization error")

ax[1].plot([o[0] for o in obs_list], label="||M||_F")
ax[1].plot([o[1] for o in obs_list], label="||W||_F")
ax[1].legend()
ax[1].set_title("Parameter norms")
plt.show()

MW = np.linalg.solve(M_lpd + 1e-5 * np.eye(M_lpd.shape[0]), W_lpd)

MW = MW / np.linalg.norm(MW, axis=1, keepdims=True)

plt.figure(figsize=(6, 4))
plt.imshow(MW, aspect="auto", cmap="bwr")
plt.colorbar(label="Weight strength")
plt.title(r"$M^{-1}W$ from LPD")
plt.xlabel("Input dimension (x,y,z)")
plt.ylabel("Latent component")
plt.show()

In [ ]:
eps = 1e-5
MW = np.linalg.solve(M_lpd + eps * np.eye(W_lpd.shape[0]), W_lpd)
MW = MW / np.linalg.norm(MW, axis=1, keepdims=True)

X_fd, Y_fd = finite_difference_vectors(trajs[:, N_stat:])

Z = np.maximum(MW @ X_fd, 0)
for i in range(Z.shape[0]):
    Z[i, :] = Z[i, :] / np.max(np.abs(Z[i, :]))

plt.figure()
plt.plot(Z.T)
plt.xlabel("Time")
plt.ylabel("Latent Components")
plt.title("LPD Latent Components over Time")
plt.show()

labels = np.argmax(Z, axis=0)
magnitude = Z[labels, np.arange(Z.shape[1])]
m_act = 1 / (1 + np.exp(-10 * (np.abs(magnitude) - 0.4)))
m_act = np.clip(m_act**1, 0.2, 1.0)

plt.figure()
plt.plot(m_act)
plt.xlabel("Time")
plt.ylabel("Activation Magnitude")
plt.title("LPD Activation Magnitude over Time")
plt.show()

In [ ]:
colors = [
    "royalblue",
    "firebrick",
    "forestgreen",
    "darkorange",
]

N = 3

A = np.ones((N, N)) * 1.5
np.fill_diagonal(A, 1.0)
A[0, 1] = 0.7
A[1, 2] = 0.7
A[2, 0] = 0.7


def glv_velocity(x):
    return x * (1 - A @ x)

plane_normal = np.array([1.0, 1.0, 1.0])
plane_normal /= np.linalg.norm(plane_normal)

grid = np.linspace(0.0, 1.0, 10)
points = []
for x in grid:
    for y in grid:
        z = 1.0 - x - y
        if -0.0 <= z <= 1.0:
            points.append([x, y, z])
points = np.array(points)

V = np.array([glv_velocity(p) for p in points])
V_plane = V - (V @ plane_normal)[:, None] * plane_normal
norms = np.linalg.norm(V_plane, axis=1)
mask = norms > 1e-8
points = points[mask]
V_plane = V_plane[mask] / norms[mask][:, None]

fig = plt.figure(figsize=(8 / 2.54, 8 / 2.54))
ax = fig.add_subplot(111, projection="3d")

ax.quiver(
    points[:, 0],
    points[:, 1],
    points[:, 2],
    V_plane[:, 0],
    V_plane[:, 1],
    V_plane[:, 2],
    length=0.08,
    normalize=True,
    linewidth=0.7,
    color="0.45",
    arrow_length_ratio=0.35,
)

start = 0
end = 3000
traj_plot = trajs[:, N_stat : (N_stat + end)]
labels_p = labels[start : end - 1]
m_act_p = m_act[start : end - 1]

points_seg = traj_plot.T.reshape(-1, 1, 3)
segments = np.concatenate([points_seg[:-1], points_seg[1:]], axis=1)

colors_rgba = [mcolors.to_rgba(colors[l], alpha=a) for l, a in zip(labels_p, m_act_p)]

lc = Line3DCollection(segments, colors=colors_rgba, linewidths=3, zorder=5)
ax.add_collection(lc)

quiver_pos = [0.75, 0.75, 0.25, 0.25]

ax_q = fig.add_axes(
    [
        ax.get_position().x0 + quiver_pos[0] * ax.get_position().width,
        ax.get_position().y0 + quiver_pos[1] * ax.get_position().height,
        quiver_pos[2] * ax.get_position().width,
        quiver_pos[3] * ax.get_position().height,
    ],
    projection="3d",
)

lim = 1.1
ax_q.set_xlim([-lim, lim])
ax_q.set_ylim([-lim, lim])
ax_q.set_zlim([-lim, lim])

for i in range(MW.shape[0]):
    ax_q.quiver(
        0,
        0,
        0,
        MW[i, 0],
        MW[i, 1],
        MW[i, 2],
        color=colors[i % len(colors)],
        linewidth=2.5,
        arrow_length_ratio=0.3,
    )

ax_q.set_proj_type("ortho")

ax_q.set_xticks([])
ax_q.set_yticks([])
ax_q.set_zticks([])

ax_q.tick_params(labelsize=9)
ax_q.text2D(0.05, -0.1, r"$\delta x_1$", fontsize=10)
ax_q.text2D(-0.08, -0.1, r"$\delta x_2$", fontsize=10)
ax_q.text2D(-0.15, -0.01, r"$\delta x_3$", fontsize=10)

ax_q.set_box_aspect([1, 1, 1])
ax_q.view_init(elev=25, azim=35)

ax_q.grid(False)
ax_q.xaxis.pane.set_alpha(0.0)
ax_q.yaxis.pane.set_alpha(0.0)
ax_q.zaxis.pane.set_alpha(0.0)

ax_q.text2D(
    0.45, 0.95, r"$V$", transform=ax_q.transAxes, fontsize=10, fontweight="bold"
)

saddles = np.eye(3)
ax.scatter(
    saddles[:, 0],
    saddles[:, 1],
    saddles[:, 2],
    marker="x",
    color="black",
    s=45,
    linewidths=2,
    depthshade=False,
    zorder=10,
)

ax.set_xlim(-0.02, 1.02)
ax.set_ylim(-0.02, 1.02)
ax.set_zlim(-0.02, 1.02)
ax.set_box_aspect([1, 1, 1])
ax.set_xticks([])
ax.set_yticks([])
ax.set_zticks([])
ax.grid(False)

ax.text(0.5, 1.2, -0.0, r"$x_1$", fontsize=12, ha="center")
ax.text(1.2, 0.6, -0, r"$x_2$", fontsize=12, ha="center")
ax.text(2, 1, 1.2, r"$x_3$", fontsize=12, ha="center")

ax.text(1.25, 0.15, 0, r"$(1,0,0)$", fontsize=10, ha="center")
ax.text(0.1, 1.2, 0, r"$(0,1,0)$", fontsize=10, ha="center")
ax.text(0, 0, 1.1, r"$(0,0,1)$", fontsize=10, ha="center")

ax.view_init(elev=28, azim=45)
plt.savefig(os.path.join(RESULTS_DIR, "fig2b_glv.png"), dpi=1000, bbox_inches="tight")
plt.show()

## Two-saddle flow on the sphere (Fig. 2c)

In [ ]:
# Divergence-free flow on the sphere from the stream function (Appendix I.4)
def velocity_on_sphere(x):
    theta = np.arccos(x[2])
    phi = np.arctan2(x[1], x[0])

    sin_theta = np.sin(theta)
    cos_theta = np.cos(theta)
    dpsi_dphi = -2 * sin_theta**2 * np.sin(2 * phi)
    dpsi_dtheta = 2 * sin_theta * cos_theta * np.cos(2 * phi)

    v_theta = -dpsi_dphi / (sin_theta + np.finfo(float).eps)
    v_phi = dpsi_dtheta

    e_theta = np.array([cos_theta * np.cos(phi), cos_theta * np.sin(phi), -sin_theta])
    e_phi = np.array([-np.sin(phi), np.cos(phi), 0])

    v = v_theta * e_theta + v_phi * e_phi
    return v


def simulate_trajectory(T=1000, dt=0.1, sigma=0.05, seed=None):
    if seed is not None:
        np.random.seed(seed)

    N = int(T / dt)
    x = np.array([1.0, 0.0, 0.0])
    traj = np.zeros((N + 1, 3))
    traj[0] = x
    dW = np.sqrt(dt) * np.random.randn(N, 3)

    for k in range(N):
        v = velocity_on_sphere(x)
        P = np.eye(3) - np.outer(x, x)
        noise_term = sigma * P @ dW[k]
        x_new = x + v * dt + noise_term
        x = x_new / np.linalg.norm(x_new)
        traj[k + 1] = x

    return traj


def fibonacci_sphere(samples=200):
    points = []
    offset = 2.0 / samples
    increment = np.pi * (3.0 - np.sqrt(5.0))

    for i in range(samples):
        z = 1 - (i + 0.5) * offset
        r = np.sqrt(1 - z * z)
        phi = i * increment
        x = r * np.cos(phi)
        y = r * np.sin(phi)
        points.append([x, y, z])

    points.append([0, 0, 1])
    points.append([0, 0, -1])

    return np.array(points)

positions = fibonacci_sphere(samples=160)
vectors = np.array([velocity_on_sphere(x) for x in positions])
vectors = vectors / (np.linalg.norm(vectors, axis=1, keepdims=True) + 1e-9) * 0.2

traj = simulate_trajectory(T=500, dt=0.01, sigma=0.05, seed=2819)

elev, azim = 25, 45

view_vector = np.array(
    [
        np.cos(np.radians(elev)) * np.cos(np.radians(azim)),
        np.cos(np.radians(elev)) * np.sin(np.radians(azim)),
        np.sin(np.radians(elev)),
    ]
)
dot_products = positions @ view_vector
visible = dot_products > 0

positions_vis = positions[visible]
vectors_vis = vectors[visible]

fig = plt.figure(figsize=(16 / 2.54, 16 / 2.54))
ax = fig.add_subplot(111, projection="3d")

phi, theta = np.mgrid[0 : np.pi : 200j, 0 : 2 * np.pi : 200j]
x_s = np.sin(phi) * np.cos(theta)
y_s = np.sin(phi) * np.sin(theta)
z_s = np.cos(phi)
ax.plot_surface(
    x_s, y_s, z_s, rstride=4, cstride=4, color="lightgray", alpha=0.1, edgecolor="none"
)

ax.quiver(
    positions_vis[:, 0],
    positions_vis[:, 1],
    positions_vis[:, 2],
    vectors_vis[:, 0],
    vectors_vis[:, 1],
    vectors_vis[:, 2],
    color="blue",
    linewidth=1.5,
    arrow_length_ratio=0.5,
    label="Velocity Field",
)

ax.plot(
    traj[-5000:, 0],
    traj[-5000:, 1],
    traj[-5000:, 2],
    color="red",
    linewidth=1,
    alpha=0.6,
    label="Trajectory",
)

ax.set_box_aspect([1, 1, 1])
ax.set_xlim([-1.2, 1.2])
ax.set_ylim([-1.2, 1.2])
ax.set_zlim([-1.2, 1.2])
ax.set_xticks([-1, 0, 1])
ax.set_yticks([-1, 0, 1])
ax.set_zticks([-1, 0, 1])

ax.tick_params(axis="x", labelsize=12)
ax.tick_params(axis="y", labelsize=12)
ax.tick_params(axis="z", labelsize=12)

ax.set_xlabel("X", fontsize=12)
ax.set_ylabel("Y", fontsize=12)
ax.set_zlabel("Z", fontsize=12)

ax.grid(False)

ax.grid(False)

ax.view_init(elev=elev, azim=azim)
ax.legend(fontsize=12)

plt.show()


def sphere_sde_simulation(seed=None):
    if seed is not None:
        np.random.seed(seed)

    T = 1_000
    dt = 0.05
    N = int(T / dt)
    sigma = 0.05

    theta0 = 0
    phi0 = 0
    x = np.array(
        [np.sin(theta0) * np.cos(phi0), np.sin(theta0) * np.sin(phi0), np.cos(theta0)]
    )

    traj = np.zeros((3, N + 1))
    traj[:, 0] = x

    dW = np.sqrt(dt) * np.random.randn(3, N)

    for k in range(N):
        v = velocity_on_sphere(x)
        P = np.eye(3) - np.outer(x, x)
        noise_term = sigma * P @ dW[:, k]
        x_new = x + v * dt + noise_term
        x = x_new / np.linalg.norm(x_new)
        traj[:, k + 1] = x

    fig = plt.figure(figsize=(8, 6))
    ax = fig.add_subplot(111, projection="3d")

    u, v = np.mgrid[0 : 2 * np.pi : 100j, 0 : np.pi : 100j]
    sx = np.cos(u) * np.sin(v)
    sy = np.sin(u) * np.sin(v)
    sz = np.cos(v)

    theta = v
    phi = u
    psi = (np.sin(theta) ** 2) * np.cos(2 * phi)

    norm_psi = (psi - psi.min()) / (psi.max() - psi.min())

    ax.plot_surface(
        sx,
        sy,
        sz,
        facecolors=plt.cm.coolwarm(norm_psi),
        rstride=1,
        cstride=1,
        linewidth=0,
        antialiased=False,
        alpha=0.7,
        shade=True,
    )

    ax.plot(traj[0], traj[1], traj[2], color="red", linewidth=2.5, label="Trajectory")
    ax.scatter(*traj[:, 0], color="blue", s=100, label="Start")
    ax.scatter(*traj[:, -1], color="green", s=100, label="End")

    for i in range(0, len(traj[0]) - 1000, 1000):
        ax.quiver(
            traj[0, i],
            traj[1, i],
            traj[2, i],
            traj[0, i + 1] - traj[0, i],
            traj[1, i + 1] - traj[1, i],
            traj[2, i + 1] - traj[2, i],
            length=0.1,
            color="black",
            arrow_length_ratio=0.3,
        )

    ax.set_xlim([-1.5, 1.5])
    ax.set_ylim([-1.5, 1.5])
    ax.set_zlim([-1.5, 1.5])
    ax.set_xlabel("x", fontsize=14)
    ax.set_ylabel("y", fontsize=14)
    ax.set_zlabel("z", fontsize=14)
    ax.view_init(elev=30, azim=45)
    ax.legend(fontsize=12)
    ax.set_box_aspect([1, 1, 1])
    ax.grid(False)

    plt.show()

    return traj

traj = sphere_sde_simulation(seed=35885)

In [ ]:
N_stat = 2000

X_fd, Y_fd = finite_difference_vectors(traj[:, N_stat:])

N_samples = 12000

X = X_fd[:, :N_samples]
Y = Y_fd[:, :N_samples]

k = 4

ONLINE = True
OFFLINE = not ONLINE

if OFFLINE:
    print("Running offline LPD...")
    Z, (W_lpd, M_lpd), (err_list, obs_list) = lpd_offline(
        X,
        Y,
        k=k,
        relu=True,
        eps=1e-5,
        Nmax=3000,
        Nz_max=100,
        toll=1e-10,
        eta_z=0.4,
        eta_W=0.4,
        eta_M=0.2,
    )
else:
    print("Running online LPD...")
    Z, (W_lpd, M_lpd), (err_list, obs_list) = lpd_online(
        X,
        Y,
        k=k,
        relu=True,
        Nz_max=200,
        toll=1e-10,
        eps=1e-5,
        eta_z=0.2,
    )

plt.figure(figsize=(10, 5))
plt.plot(Z.T)
plt.xlabel("Time")
plt.ylabel("Latent Components")
plt.legend([f"Component {i+1}" for i in range(Z.shape[0])])
plt.show()

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot([e[0] for e in err_list], label="err_M")
ax[0].plot([e[1] for e in err_list], label="err_W")
ax[0].legend()
ax[0].set_title("LPD optimization error")

ax[1].plot([o[0] for o in obs_list], label="||M||_F")
ax[1].plot([o[1] for o in obs_list], label="||W||_F")
ax[1].legend()
ax[1].set_title("Parameter norms")
plt.show()

MW = np.linalg.solve(M_lpd + 1e-5 * np.eye(M_lpd.shape[0]), W_lpd)
MW = MW / np.linalg.norm(MW, axis=1, keepdims=True)

plt.figure(figsize=(6, 4))
plt.imshow(MW, aspect="auto", cmap="bwr", vmin=-1, vmax=1)
plt.colorbar(label="Weight strength")
plt.title(r"$M^{-1}W$ from LPD")
plt.xlabel("Input dimension (x,y,z)")
plt.ylabel("Latent component")
plt.show()

In [ ]:
MW = np.linalg.solve(M_lpd + 1e-5 * np.eye(W_lpd.shape[0]), W_lpd)
MW = MW / np.linalg.norm(MW, axis=1, keepdims=True)

Z = np.maximum(MW @ X_fd, 0)
for i in range(Z.shape[0]):
    Z[i, :] = Z[i, :] / np.max(np.abs(Z[i, :]))

plt.plot(Z.T)
plt.xlim(100, 7000)

plt.figure()
labels = np.argmax(Z, axis=0)
magnitude = Z[labels, np.arange(Z.shape[1])]
m_act = 1 / (1 + np.exp(-10 * (np.abs(magnitude) - 0.5)))
m_act = np.clip(m_act**1, 0.1, 1.0)
plt.plot(m_act)
plt.xlim(0, 1000)

In [ ]:
colors = [
    "royalblue",
    "goldenrod",
    "forestgreen",
    "firebrick",
    "darkorange",
    "purple",
    "teal",
]

N = 3000
traj_p = traj[:, N_stat : N_stat + N]
labels_p = labels[:N]
m_act_p = m_act[:N]

positions = fibonacci_sphere(samples=240)
vectors = np.array([velocity_on_sphere(x) for x in positions])
vectors = vectors / (np.linalg.norm(vectors, axis=1, keepdims=True) + 1e-9) * 0.2

elev, azim = 10, 30
view_vector = np.array(
    [
        np.cos(np.radians(elev)) * np.cos(np.radians(azim)),
        np.cos(np.radians(elev)) * np.sin(np.radians(azim)),
        np.sin(np.radians(elev)),
    ]
)

dot_products = positions @ view_vector
visible = dot_products > 0
positions_vis = positions[visible]
vectors_vis = vectors[visible]

colors_arr = np.array(colors)
rgba = [mcolors.to_rgba(colors_arr[l], alpha=a) for l, a in zip(labels_p, m_act_p)]

fig = plt.figure(figsize=(8 / 2.54, 8 / 2.54))
ax = fig.add_subplot(111, projection="3d")

phi, theta = np.mgrid[0 : np.pi : 200j, 0 : 2 * np.pi : 200j]
x_s = np.sin(phi) * np.cos(theta)
y_s = np.sin(phi) * np.sin(theta)
z_s = np.cos(phi)
ax.plot_surface(
    x_s, y_s, z_s, rstride=4, cstride=4, color="lightgray", alpha=0.1, edgecolor="none"
)

ax.quiver(
    positions_vis[:, 0],
    positions_vis[:, 1],
    positions_vis[:, 2],
    vectors_vis[:, 0],
    vectors_vis[:, 1],
    vectors_vis[:, 2],
    color="black",
    linewidth=0.7,
    arrow_length_ratio=0.55,
    label="Velocity Field",
    zorder=1,
)

ax.scatter(traj_p[0], traj_p[1], traj_p[2], c=rgba, s=4, depthshade=False)

quiver_pos = [0.82, 0.76, 0.3, 0.3]
ax_q = fig.add_axes(
    [
        ax.get_position().x0 + quiver_pos[0] * ax.get_position().width,
        ax.get_position().y0 + quiver_pos[1] * ax.get_position().height,
        quiver_pos[2] * ax.get_position().width,
        quiver_pos[3] * ax.get_position().height,
    ],
    projection="3d",
)

colors_quiver = colors[: MW.shape[0]]
for i in range(MW.shape[0]):
    ax_q.quiver(
        0,
        0,
        0,
        MW[i, 0],
        MW[i, 1],
        MW[i, 2],
        color=colors_quiver[i % len(colors_quiver)],
        linewidth=3.0,
        arrow_length_ratio=0.3,
    )

ax_q.set_xlim([-1, 1])
ax_q.set_ylim([-1, 1])
ax_q.set_zlim([-1, 1])
ax_q.set_xticks([])
ax_q.set_yticks([])
ax_q.set_zticks([])
ax_q.tick_params(labelsize=9)

ax_q.text2D(
    0.85,
    0.05,
    r"$\delta x_1$",
    transform=ax_q.transAxes,
    va="center",
    ha="center",
    fontsize=10,
)
ax_q.text2D(
    -0.05,
    0.5,
    r"$\delta x_3$",
    transform=ax_q.transAxes,
    rotation=90,
    va="center",
    ha="center",
    fontsize=10,
)
ax_q.text2D(
    0.3,
    0.0,
    r"$\delta x_2$",
    transform=ax_q.transAxes,
    va="center",
    ha="center",
    fontsize=10,
)

ax_q.set_box_aspect([1.2, 1.2, 1.2])
ax_q.view_init(elev=10, azim=25)
ax_q.grid(False)
ax_q.xaxis.pane.set_alpha(0.0)
ax_q.yaxis.pane.set_alpha(0.0)
ax_q.zaxis.pane.set_alpha(0.0)
ax_q.text2D(0.35, 0.8, r"$V$", transform=ax_q.transAxes, fontsize=10, fontweight="bold")

ax.set_box_aspect([1, 1, 1])
ax.set_xlim([-1.0, 1.0])
ax.set_ylim([-1.0, 1.0])
ax.set_zlim([-1, 1])
ax.set_xticks([-1, 1])
ax.set_yticks([-1, 1])
ax.set_zticks([-1, 1])

ax.tick_params(axis="x", labelsize=12)
ax.tick_params(axis="y", labelsize=12)
ax.tick_params(axis="z", labelsize=12)

ax.text2D(
    0.82, 0.07, r"$x_1$", transform=ax.transAxes, va="center", ha="center", fontsize=12
)
ax.text2D(
    0.35, 0.05, r"$x_2$", transform=ax.transAxes, va="center", ha="center", fontsize=12
)
ax.text2D(
    -0.05, 0.5, r"$x_3$", transform=ax.transAxes, va="center", ha="center", fontsize=12
)

ax.grid(False)
ax.view_init(elev=elev, azim=azim)

plt.savefig(
    os.path.join(RESULTS_DIR, "fig2c_sphere.png"), bbox_inches="tight", dpi=1000
)
plt.show()

## CCA benchmark with a Gaussian latent variable (Fig. 4)

In [ ]:
n_samples = 40
n_features_x = 10
n_features_y = 10
latent_dim = 1
noise_std = 0.1

Z_true = np.random.randn(n_samples, latent_dim)

A = np.random.randn(n_features_x, latent_dim)
B = np.random.randn(n_features_y, latent_dim)

X = Z_true @ A.T + noise_std * np.random.randn(n_samples, n_features_x)
Y = Z_true @ B.T + noise_std * np.random.randn(n_samples, n_features_y)

X = StandardScaler().fit_transform(X)
Y = StandardScaler().fit_transform(Y)

cca = CCA(n_components=latent_dim)
X_c, Y_c = cca.fit_transform(X, Y)
Z_cca = X_c.T

In [ ]:
k = latent_dim
X_data = X.T
Y_data = Y.T
Z_lpd, (W, M), (err, obs) = lpd_offline(
    X_data,
    Y_data,
    k=k,
    relu=False,
    eps=1e-5,
    Nmax=3000,
    Nz_max=100,
    toll=1e-10,
    eta_z=0.4,
    eta_W=0.4,
    eta_M=0.2,
)

for i in range(k):
    Z_lpd[i, :] /= np.linalg.norm(Z_lpd[i, :])

In [ ]:
Z_nn = lpd_offline(
    X_data,
    Y_data,
    k=2,
    relu=True,
    eps=1e-5,
    Nmax=3000,
    Nz_max=100,
    toll=1e-10,
    eta_z=0.4,
    eta_W=0.4,
    eta_M=0.2,
)[0]

In [ ]:
plt.plot(X_data.T)
plt.plot(Z_true, linewidth=3, label="True Latent", color="black")
plt.xlabel("Sample Index")
plt.ylabel("X Values")
plt.show()

In [ ]:
plt.plot(Y_data.T)
plt.plot(Z_true, linewidth=3, label="True Latent", color="black")
plt.xlabel("Sample Index")
plt.ylabel("Y Values")
plt.show()

In [ ]:
Z_true = Z_true / np.max(np.abs(Z_true))
Z_cca = Z_cca / np.max(np.abs(Z_cca))
Z_lpd = Z_lpd / np.max(np.abs(Z_lpd))

plt.plot(Z_true, label="True Latent Z", linewidth=2)
plt.plot(Z_cca.T, label="CCA Latent Z", linestyle="--")
plt.plot(Z_lpd.T, label="LPD Latent Z", linestyle=":")

plt.legend()

plt.xlabel("Sample Index")
plt.ylabel("Z")

In [ ]:
Z_nn = Z_nn / np.max(np.abs(Z_nn), axis=1, keepdims=True)
Z_true = Z_true / np.max(np.abs(Z_true))

plt.plot(Z_nn[0, :], label="NN Latent Z", linestyle="-.")
plt.plot(-Z_nn[1, :], linestyle="-.")
plt.plot(Z_true, linewidth=3, label="True Latent", color="black")

In [ ]:
Z_true_norm = Z_true / np.max(np.abs(Z_true))
Z_cca_norm = Z_cca / np.max(np.abs(Z_cca))
Z_lpd_norm = Z_lpd / np.max(np.abs(Z_lpd))
Z_nn_norm = Z_nn / np.max(np.abs(Z_nn), axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(18 / 2.54, 6 / 2.54))

axes[0].plot(Z_cca_norm.T, label="CCA", color="firebrick", lw=5, alpha=1)
axes[0].plot(Z_lpd_norm.T, label=r"$Z$", color="royalblue", lw=2, alpha=1)
axes[0].plot(
    Z_true_norm,
    label=r"$Z_{\text{Latent}}$",
    linestyle="--",
    linewidth=2,
    color="black",
)
axes[0].set_xlabel("Sample Index")
axes[0].set_ylabel("Z")
axes[0].legend(fontsize=10)
axes[0].set_yticks([])
axes[0].legend(fontsize=10, loc="center right")
axes[0].set_yticks([])
axes[0].set_xlim(0, 60)
axes[0].set_xticks([0, 20, 40])

colors = ["royalblue", "firebrick", "darkorange", "indigo", "goldenrod", "forestgreen"]

axes[1].plot(Z_nn_norm[0, :], linewidth=3, label=r"$Z_1$", color="forestgreen")
axes[1].plot(-Z_nn_norm[1, :], linewidth=3, color="darkorange", label=r"$-Z_2$")
axes[1].plot(
    Z_true_norm,
    linewidth=2,
    linestyle="--",
    label=r"$Z_{\text{Latent}}$",
    color="black",
)
axes[1].set_xlabel("Sample Index")
axes[1].set_ylabel("Z")
axes[1].legend(fontsize=10, loc="center right")
axes[1].set_yticks([])
axes[1].set_xlim(0, 60)
axes[1].set_xticks([0, 20, 40])

axes[1].text(42, 0.7, "Non-negative", fontsize=10, color="black")
axes[0].text(42, 0.7, "Unconstrained", fontsize=10, color="black")
axes[1].text(-3, 1, "b)", fontsize=10, color="black")
axes[0].text(-3, 1, "a)", fontsize=10, color="black")

plt.tight_layout()
plt.savefig(
    os.path.join(RESULTS_DIR, "cca_comparison.png"), dpi=1000, bbox_inches="tight"
)

In [ ]:
G_true = Z_true @ Z_true.T / n_samples
G_cca = Z_cca.T @ Z_cca / n_samples
G_lpd = Z_lpd.T @ Z_lpd / n_samples

plt.figure(figsize=(12, 4))
plt.subplot(1, 3, 1)
plt.imshow(G_true, cmap="viridis", interpolation="nearest")
plt.colorbar(label="G_true Values")
plt.title("True Correlation Matrix G_true")
plt.subplot(1, 3, 2)
plt.imshow(G_cca, cmap="viridis", interpolation="nearest")
plt.colorbar(label="G_cca Values")
plt.title("CCA Correlation Matrix G_cca")
plt.subplot(1, 3, 3)
plt.imshow(G_lpd, cmap="viridis", interpolation="nearest")
plt.colorbar(label="G_lpd Values")
plt.title("LPD Correlation Matrix G_lpd")
plt.show()

## Model selection and validation (Fig. 6)

In [ ]:
N_shc = 3
A_shc = np.ones((N_shc, N_shc)) * 1.5
np.fill_diagonal(A_shc, 1.0)

A_shc[0, 1] = 0.7
A_shc[1, 2] = 0.7
A_shc[2, 0] = 0.7

A = A_shc
N = N_shc

u_unstable = np.array([(0, 1, -1), (-1, 0, 1), (1, -1, 0)])

In [ ]:
# Euler-Maruyama simulation of the gLV system (Appendix I.2)
def simulation_glv_shc(T, dt, sigma, A_shc, x0=[0.5, 0.5, 0], plot=True):

    num_steps = int(T / dt)
    X = np.zeros((N, num_steps))
    X[:, 0] = x0

    for k in range(1, num_steps):

        dx = X[:, k - 1] * (1 - A @ X[:, k - 1])

        dx += sigma * np.random.randn(N)

        X[:, k] = X[:, k - 1] + dx * dt

        X[:, k] = np.clip(X[:, k], 0, None)

    trajs = X

    t_eval = np.arange(0, T, dt)

    if plot == True:

        plt.figure(figsize=(16 / 2.54, 5 / 2.54))
        for i in range(N_shc):
            plt.plot(t_eval, X[i], label=f"S{i+1}")
        for j in range(N_shc, N):
            plt.plot(t_eval, X[j], color="gray", alpha=0.3)
        plt.xlabel("Time")
        plt.xlim(800, 1000)
        plt.ylabel("Population")
        plt.xticks([])
        plt.yticks([0, 1])
        plt.legend()
        plt.show()

    return trajs


def split(traj, N_stat, N_data_train, N_data_test):
    X_fd, Y_fd = finite_difference_vectors(traj[:, N_stat:])
    X = X_fd[:, :N_data_train]
    Y = Y_fd[:, :N_data_train]

    X_test = X_fd[:, N_data_train : N_data_train + N_data_test]
    Y_test = Y_fd[:, N_data_train : N_data_train + N_data_test]

    return X, Y, X_test, Y_test


def train_lpd(X, Y, k, relu=True, plot=True):
    Z, (W_lpd, M_lpd), (err_list, obs_list) = lpd_offline(
        X,
        Y,
        k=k,
        relu=relu,
        eps=1e-5,
        Nmax=3000,
        Nz_max=100,
        toll=1e-10,
        eta_z=0.4,
        eta_W=0.4,
        eta_M=0.2,
    )
    MW = np.linalg.solve(M_lpd + 1e-5 * np.eye(M_lpd.shape[0]), W_lpd)

    MW = MW / np.linalg.norm(MW, axis=1, keepdims=True)

    if plot == True:
        plt.figure(figsize=(6, 4))
        plt.imshow(MW, aspect="auto", cmap="bwr")
        plt.colorbar(label="Weight strength")
        plt.title(r"$M^{-1}W$ from LPD")
        plt.xlabel("Input dimension (x,y,z)")
        plt.ylabel("Latent component")
        plt.show()

    return MW


def test_LPD(MW, u_unstable, k=2):
    alignment = np.zeros((MW.shape[0], u_unstable.shape[0]))
    for i in range(MW.shape[0]):
        for j in range(u_unstable.shape[0]):
            alignment[i, j] = np.dot(MW[i, :], u_unstable[j, :]) / (
                np.linalg.norm(MW[i, :]) * np.linalg.norm(u_unstable[j, :])
            )
    accuracy = 0
    for i in range(k):
        max_idx = np.argmax(np.abs(alignment[i, :]))
        accuracy += np.abs(alignment[i, max_idx])
    accuracy = accuracy / k
    return accuracy, alignment


# Precision/recall of learned vs. ground-truth directions (Eq. 39)
def signed_matching_metrics(M, U):
    """
    Matching-based metrics where sign matters.
    Negative cosine similarities are treated as incorrect (zero reward).
    """

    M = M / np.linalg.norm(M, axis=1, keepdims=True)
    U = U / np.linalg.norm(U, axis=1, keepdims=True)

    C = M @ U.T

    C = np.maximum(C, 0.0)

    row_ind, col_ind = linear_sum_assignment(-C)
    matched_scores = C[row_ind, col_ind]

    k, k_true = M.shape[0], U.shape[0]

    precision = matched_scores.sum() / k
    recall = matched_scores.sum() / k_true
    sym_acc = matched_scores.sum() / max(k, k_true)

    f1 = 2 * precision * recall / (precision + recall + 1e-12)

    return {
        "symmetric_accuracy": sym_acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }

In [ ]:
T = 2000
dt = 0.1
sigma = 0.2

traj = simulation_glv_shc(T, dt, sigma, A_shc)

X_train, Y_train, X_test, Y_test = split(
    traj, N_stat=int(0 // dt), N_data_train=int(2000 // dt), N_data_test=int(1 // dt)
)

In [ ]:
os.makedirs(os.path.join(DATA_DIR, "model_selection"), exist_ok=True)
sigma_list = [0.005, 0.01, 0.05, 0.1, 0.5]
k_list = [1, 2, 3, 4, 5, 6]
N_trials = 10

df = pd.DataFrame(
    columns=[
        "n",
        "sigma",
        "k",
        "symmetric_accuracy",
        "precision",
        "recall",
        "f1",
        "obj",
    ]
)

for n in range(N_trials):

    for sigma in sigma_list:
        print(f"Simulating GLV SHC with sigma={sigma}...")
        traj = simulation_glv_shc(1410, dt, sigma, A_shc, plot=False)

        X_train, Y_train, X_test, Y_test = split(
            traj,
            N_stat=int(10 // dt),
            N_data_train=int(1000 // dt),
            N_data_test=int(400 // dt),
        )

        X = X_test
        Y = Y_test
        eps = 1e-5
        Nx = X.shape[0]
        Ny = Y.shape[0]

        G = (
            X.T
            @ np.linalg.inv(X @ X.T + eps * np.eye(Nx))
            @ (X @ Y.T)
            @ np.linalg.inv(Y @ Y.T + eps * np.eye(Ny))
            @ Y
        )

        gt = G @ (G.T)
        gt = gt / np.linalg.norm(gt, "fro")

        print(f"Training LPD on data with sigma={sigma}...")

        for k in k_list:

            MW = train_lpd(X_train, Y_train, k=k, plot=False)
            metrics = signed_matching_metrics(MW, u_unstable)

            Z = np.maximum(MW @ X_test, 0)
            for i in range(Z.shape[0]):
                Z[i, :] = Z[i, :] / np.linalg.norm(Z[i, :])

            exp = Z.T @ Z / Z.shape[1]
            exp = exp / np.linalg.norm(exp, "fro")

            obj = np.linalg.norm(gt - exp, "fro") ** 2 / gt.shape[0]

            np.savez(
                os.path.join(
                    DATA_DIR,
                    "model_selection",
                    f"LPD_glv_shc_n_{n}_sigma_{sigma}_k_{k}.npz",
                ),
                MW=MW,
                metrics=metrics,
                obj=obj,
            )

            df.loc[len(df)] = {
                "n": n,
                "sigma": sigma,
                "k": k,
                "symmetric_accuracy": metrics["symmetric_accuracy"],
                "precision": metrics["precision"],
                "recall": metrics["recall"],
                "f1": metrics["f1"],
                "obj": obj,
            }

df.to_csv(os.path.join(DATA_DIR, "model_selection.csv"), index=False)

**TODO before release:** the next cell needs `objective_evaluation` (objective of Eq. 35 with the $k$ row-rescaling factors, Appendix J) and the test sets stacked per $\sigma$ (`X_test`, `Y_test`).

In [ ]:
def objective_evaluation(X, Y, Z):
    raise NotImplementedError("Add the row-rescaled objective of Eq. (35).")

files = sorted(
    f
    for f in os.listdir(os.path.join(DATA_DIR, "model_selection"))
    if f.endswith(".npz")
)
df = pd.DataFrame(
    columns=[
        "n",
        "sigma",
        "k",
        "symmetric_accuracy",
        "precision",
        "recall",
        "f1",
        "obj",
        "a",
    ]
)
for i, file in enumerate(files):
    if i % 10 == 0:
        print(f"Processing file {i+1}/{len(files)}: {file}")

    data = np.load(os.path.join(DATA_DIR, "model_selection", file), allow_pickle=True)
    MW = data["MW"]
    metrics = data["metrics"].item()
    obj = data["obj"]

    filename = file

    pattern = r"_n_(?P<n>\d+)_sigma_(?P<sigma>[0-9.]+)_k_(?P<k>\d+)"
    match = re.search(pattern, filename)

    if match:
        n = int(match.group("n"))
        sigma = float(match.group("sigma"))
        k = int(match.group("k"))
        print(f"n={n}, sigma={sigma}, k={k}")
    else:
        raise ValueError("Filename format not recognized")

    X = X_test[sigma == sigma_list, :, :].reshape(X_test.shape[1], X_test.shape[2])
    Y = Y_test[sigma == sigma_list, :, :].reshape(Y_test.shape[1], Y_test.shape[2])

    obj, a_est = objective_evaluation(X, Y, np.maximum(MW @ X, 0))

    df.loc[len(df)] = {
        "n": n,
        "sigma": sigma,
        "k": k,
        "symmetric_accuracy": metrics["symmetric_accuracy"],
        "precision": metrics["precision"],
        "recall": metrics["recall"],
        "f1": metrics["f1"],
        "obj": obj,
        "a": a_est,
    }

df.to_csv(os.path.join(DATA_DIR, "model_selection_rescaled.csv"), index=False)

In [ ]:
df_grouped = df.groupby(["sigma", "k"]).mean(numeric_only=True).reset_index()

In [ ]:
plt.figure(figsize=(8 / 2.54, 6 / 2.54))
for sigma in sigma_list:
    df_sigma = df_grouped[df_grouped["sigma"] == sigma]
    plt.plot(
        df_sigma["k"], df_sigma["precision"], marker="o", label=r"$σ$={}".format(sigma)
    )
plt.xlabel(r"Number of components k")
plt.ylabel("Precision")
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(8 / 2.54, 6 / 2.54))
for sigma in sigma_list:
    df_sigma = df_grouped[df_grouped["sigma"] == sigma]
    plt.plot(
        df_sigma["k"], df_sigma["recall"], marker="o", label=r"$σ$={}".format(sigma)
    )
plt.xlabel(r"Number of components k")
plt.ylabel("Recall")
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(8 / 2.54, 6 / 2.54))
for sigma in sigma_list:
    df_sigma = df_grouped[df_grouped["sigma"] == sigma]
    plt.plot(df_sigma["k"], df_sigma["obj"], marker="o", label=r"$σ$={}".format(sigma))
plt.xlabel(r"Number of components k")
plt.ylabel("Objective")
plt.legend()
plt.show()

In [ ]:
colors = ["royalblue", "firebrick", "seagreen", "darkorange"]
sigmas = [0.01, 0.05, 0.1]

fig, axes = plt.subplots(1, 3, figsize=(18 / 2.54, 6 / 2.54), sharex=True)

metrics = ["obj", "precision", "recall"]
ylabels = ["Objective", "Precision", "Recall"]
panel_labels = ["a)", "b)", "c)"]

for ax, metric, ylabel in zip(axes, metrics, ylabels):
    for i, sigma in enumerate(sigmas):
        df_sigma = df_grouped[df_grouped["sigma"] == sigma]
        ax.plot(
            df_sigma["k"],
            df_sigma[metric],
            marker="o",
            label=rf"$\sigma$={sigma}",
            color=colors[i],
        )

    ax.set_xlabel("Output components $k$")
    ax.set_ylabel(ylabel)
    ax.margins(x=0.02)
    ax.set_xticks(range(1, 7))

    if metric == "obj":
        formatter = mticker.ScalarFormatter(useMathText=True)
        formatter.set_powerlimits((0, 0))
        ax.yaxis.set_major_formatter(formatter)

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(
    handles,
    labels,
    loc="upper center",
    ncol=len(sigmas),
    frameon=False,
    bbox_to_anchor=(0.5, 1.1),
)

for i, (ax, panel) in enumerate(zip(axes, panel_labels)):
    bbox = ax.get_position()
    x_offset = [-0.09, -0.05, 0.0][i]
    fig.text(
        bbox.x0 + x_offset,
        bbox.y1 + 0.01,
        panel,
        fontsize=10,
        fontweight="bold",
        ha="left",
        va="bottom",
    )

plt.subplots_adjust(wspace=0.25, top=0.80, bottom=0.18, left=0.10, right=0.98)

plt.savefig(
    os.path.join(RESULTS_DIR, "model_selection_validation.png"),
    dpi=1000,
    bbox_inches="tight",
)

## Ablation: finite differences and non-negativity (Fig. 9)

In [ ]:
os.makedirs(os.path.join(DATA_DIR, "metrics"), exist_ok=True)
sigma_list = [0.01, 0.05, 0.1]
k_list = [1, 2, 3, 4, 5, 6]
RELU = [True, False]
fd_list = [True, False]
N_trials = 10
dt = 0.1

results = []


# One ablation run: relu toggles non-negativity, fd toggles finite differences
def run_trial(n, fd, sigma, k, relu):
    print(
        f"Simulating GLV SHC with sigma={sigma}, trial={n}, k={k}, relu={relu}, fd={fd}"
    )

    traj = simulation_glv_shc(1410, dt, sigma, A_shc, plot=False)

    if fd:
        X_train, Y_train, X_test, Y_test = split(
            traj,
            N_stat=int(10 / dt),
            N_data_train=int(1000 / dt),
            N_data_test=int(400 / dt),
        )
        Z_true = np.maximum(u_unstable @ X_test, 0)
    else:
        X_train = traj[:, int(10 / dt) : int(1010 / dt)]
        Y_train = traj[:, int(10 / dt) + 1 : int(1010 / dt) + 1]
        X_test = traj[:, int(1010 / dt) : int(1410 / dt)]
        Y_test = traj[:, int(1010 / dt) + 1 : int(1410 / dt) + 1]

        X_test_true, _ = finite_difference_vectors(X_test)
        X_test = X_test[:, : X_test.shape[1] - 2]
        Y_test = Y_test[:, : Y_test.shape[1] - 2]

        Z_true = np.maximum(u_unstable @ X_test_true, 0)

    MW = train_lpd(X_train, Y_train, k=k, relu=relu, plot=False)

    np.savez(
        os.path.join(
            DATA_DIR,
            "metrics",
            f"LPD_glv_shc_n_{n}_sigma_{sigma}_k_{k}_relu_{relu}_fd_{fd}.npz",
        ),
        MW=MW,
    )

    Z_model = np.maximum(MW @ X_test, 0)
    labels_true = np.argmax(Z_true, axis=0)
    labels_model = np.argmax(Z_model, axis=0)
    ari = adjusted_rand_score(labels_true, labels_model)
    nmi = normalized_mutual_info_score(labels_true, labels_model)

    return {
        "n": n,
        "sigma": sigma,
        "k": k,
        "relu": relu,
        "fd": fd,
        "ari": ari,
        "nmi": nmi,
    }

all_results = Parallel(n_jobs=-1, verbose=11)(
    delayed(run_trial)(n, fd, sigma, k, relu)
    for n in range(N_trials)
    for fd in fd_list
    for sigma in sigma_list
    for k in k_list
    for relu in RELU
)

df = pd.DataFrame(all_results)
df.to_csv(os.path.join(DATA_DIR, "metrics_glv_shc.csv"), index=False)

In [ ]:
df_grouped = (
    df.groupby(["sigma", "k", "relu", "fd"]).mean(numeric_only=True).reset_index()
)

In [ ]:
sigma = 0.1

plt.figure(figsize=(8 / 2.54, 6 / 2.54))
for relu in RELU:
    for fd in fd_list:
        df_subset = df_grouped[
            (df_grouped["sigma"] == sigma)
            & (df_grouped["relu"] == relu)
            & (df_grouped["fd"] == fd)
        ]
        label = f"ReLU={relu}, FD={fd}"
        plt.plot(df_subset["k"], df_subset["ari"], marker="o", label=label)
plt.xlabel(r"Number of components k")
plt.ylabel("Adjusted Rand Index (ARI)")
plt.legend()
plt.show()

In [ ]:
sigma = 0.1

plt.figure(figsize=(8 / 2.54, 6 / 2.54))
for relu in RELU:
    for fd in fd_list:
        df_subset = df_grouped[
            (df_grouped["sigma"] == sigma)
            & (df_grouped["relu"] == relu)
            & (df_grouped["fd"] == fd)
        ]
        label = f"ReLU={relu}, FD={fd}"
        plt.plot(df_subset["k"], df_subset["nmi"], marker="o", label=label)
plt.xlabel(r"Number of components k")
plt.ylabel("Normalized Mutual Information (NMI)")
plt.legend()
plt.show()

In [ ]:
colors = ["royalblue", "firebrick", "seagreen", "darkorange"]
sigmas = [0.01, 0.05, 0.1]
sigma_labels = [r"$\sigma = 0.01$", r"$\sigma = 0.05$", r"$\sigma = 0.1$"]
metrics = ["ari", "nmi"]

legend_labels = [
    "Local Predictive Learning",
    "Non-negative Past-Future CCA",
    "Finite Differences CCA",
    "Past-Future CCA",
]

combinations = list(itertools.product(RELU, fd_list))

fig, axes = plt.subplots(
    2, 3, figsize=(18 / 2.54, 12 / 2.54), sharex=True, sharey="row"
)

lines_for_legend = []

for col, sigma in enumerate(sigmas):
    for idx, (relu, fd) in enumerate(combinations):
        df_subset = df_grouped[
            (df_grouped["sigma"] == sigma)
            & (df_grouped["relu"] == relu)
            & (df_grouped["fd"] == fd)
        ]

        label = legend_labels[idx]

        (l1,) = axes[0, col].plot(
            df_subset["k"], df_subset["ari"], marker="o", color=colors[idx]
        )

        (l2,) = axes[1, col].plot(
            df_subset["k"], df_subset["nmi"], marker="o", color=colors[idx]
        )

        if col == 0:
            lines_for_legend.append(l1)

    axes[0, col].set_title(sigma_labels[col], fontsize=10)

axes[0, 0].set_ylabel("ARI")
axes[1, 0].set_ylabel("NMI")

for ax in axes[1, :]:
    ax.set_xlabel("k")
    ax.set_xticks([1, 2, 3, 4, 5, 6])

fig.legend(
    lines_for_legend,
    legend_labels,
    loc="upper center",
    ncol=len(legend_labels),
    bbox_to_anchor=(0.5, 1.05),
)

plt.tight_layout()
plt.savefig(
    os.path.join(RESULTS_DIR, "metrics_and_comparison.png"),
    dpi=1000,
    bbox_inches="tight",
)

In [ ]:
sigma = 0.01
metric = "nmi"
colors = ["royalblue", "firebrick", "seagreen", "darkorange"]
legend_labels = ["NN FD", "NN PF", "FD", "PF"]
combinations = list(itertools.product(RELU, fd_list))
markers = ["o", "s", "D", "^"]

fig, ax = plt.subplots(figsize=(8 / 2.54, 6 / 2.54))

lines = []

for idx, (relu, fd) in enumerate(combinations):
    df_subset = df_grouped[
        (df_grouped["sigma"] == sigma)
        & (df_grouped["relu"] == relu)
        & (df_grouped["fd"] == fd)
    ]

    (l,) = ax.plot(
        df_subset["k"],
        df_subset[metric],
        marker=markers[idx],
        color=colors[idx],
        label=legend_labels[idx],
    )
    lines.append(l)

ax.set_xlabel("Number of components k")
ax.set_ylabel("NMI")
ax.set_xticks([1, 2, 3, 4, 5, 6])
ax.set_yticks([0.0, 0.5, 1.0])

ax.legend(
    loc="lower center",
    bbox_to_anchor=(0.5, 1.02),
    ncol=len(legend_labels),
    fontsize=9,
    frameon=False,
)

plt.tight_layout()
plt.savefig(
    os.path.join(RESULTS_DIR, "ablation_nmi_sigma0.01.png"),
    dpi=1000,
    bbox_inches="tight",
)
plt.show()